In [161]:
# !tar -xzvf experiments.tar.gz

In [162]:
import os
import json
import plotly.express as px
from plotly import graph_objects as go
import numpy as np
import pandas as pd
import glob
import math

In [163]:
lstm_jsons = glob.glob('experiments/*/lstm/hparams.json')
trf_jsons = glob.glob('experiments/*/trf/hparams.json')

In [164]:
def fetch_data(
    arch,
    gram,
    vocab_size,
    states
):
    
    if arch == 'lstm':
        jsons = lstm_jsons
    elif arch == 'trf':
        jsons = trf_jsons
    elif arch == 'any':
        jsons = lstm_jsons + trf_jsons
    else:
        raise KeyError
    
    if gram not in ('pfsa', 'pcfg', 'any'):
        raise KeyError
    if vocab_size not in (1000, 5000, 'any'):
        raise KeyError
    if states not in (2, 4, 8, 16, 32, 64, 'any'):
        raise KeyError
    
    result = []
    
    for j in jsons:
        data = json.load(open(j))
        if (
            data['grammar_type'] == gram or gram == 'any'
        ) and (
            data['grammar_num_symbols'] == vocab_size or vocab_size == 'any'
        ) and (
            data['grammar_formalism_arg'] == states or states == 'any'
        ):
            result.append((
                j.split(os.path.sep)[1],
                data,
                pd.read_csv(j.replace('hparams.json', 'metrics.tsv'), sep='\t')
            ))
    
    return result

In [165]:
def best_step(df: pd.DataFrame, crit):
    if crit == 'spearman':
        col = 'spearman_weighted_avg'
        return df[df[col] == df[col].max()].iloc[-1]
    elif crit == 'ce':
        col = 'ce'
        return df[df[col] == df[col].min()].iloc[-1]
    else:
        raise KeyError

In [166]:
df = pd.DataFrame()

for experiment_num, hparams, metrics in fetch_data('any', 'any', 'any', 'any'):
    
    best_step_spearman = best_step(metrics, 'spearman')
    best_step_ce = best_step(metrics, 'ce')
    
    df = pd.concat((df, pd.DataFrame({
        'Experiment number': [int(experiment_num)],
        'Architecture': [hparams['model_type']],
        'Grammar type': [hparams['grammar_type']],
        'Vocabulary size': [hparams['grammar_num_symbols']],
        'Log number of states or non-terminals': [math.log(hparams['grammar_formalism_arg'], 2)],
        'Seed': [hparams['grammar_seed']],
        'Best step by SLAC': [best_step_spearman['step']],
        'Best step by CE': [best_step_ce['step']],
        'Mean length': [hparams['train_data_stats']['mean_length']],
        'Entropy': [hparams['grammar_actual_entropy']],
        'Best SLAC': [best_step_spearman['spearman_weighted_avg']],
        'Best CE': [best_step_ce['ce']],
        'Excess entropy': [hparams['train_data_ee']],
        'p-value sum (SLAC)': [best_step_spearman['sum_of_pvals']]
    })), ignore_index=True)
    
df['Token-wise entropy'] = df['Entropy'] / df['Mean length']
    
df = df.sort_values(by=['Experiment number', 'Architecture'])

df = df.set_index('Experiment number')

df

,Architecture,Grammar type,Vocabulary size,Log number of states or non-terminals,Seed,Best step by SLAC,Best step by CE,Mean length,Entropy,Best SLAC,Best CE,Excess entropy,p-value sum (SLAC),Token-wise entropy
Experiment number,,,,,,,,,,,,,,
1,lstm,pfsa,1000,1.0,0,1000.0,4000.0,2.970625,7.506837,0.598301,6.790781,5.978066,9.037035e-06,2.527023
1,trf,pfsa,1000,1.0,0,6000.0,5000.0,2.970625,7.506837,0.806175,2.238441,5.978066,2.657251e-06,2.527023
2,lstm,pfsa,1000,2.0,0,1800.0,1000.0,5.097094,8.016413,0.678655,6.881297,8.561621,5.500731e-07,1.572742
2,trf,pfsa,1000,2.0,0,4700.0,5800.0,5.097094,8.016413,0.720852,2.084914,8.561621,2.027965e-07,1.572742
3,lstm,pfsa,1000,3.0,0,2000.0,2300.0,8.937500,8.602941,0.707898,6.908571,10.358926,2.676136e-128,0.962567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,lstm,pfsa,1000,2.0,0,1900.0,3000.0,5.092875,8.016413,0.687040,6.879758,8.560159,1.270518e-04,1.574045
118,trf,pfsa,1000,2.0,0,10000.0,7800.0,5.092875,8.016413,0.717870,2.086285,8.560159,1.749529e-04,1.574045
119,lstm,pcfg,1000,4.0,0,1000.0,1900.0,1.317500,3.938411,0.645741,6.551202,3.481372,3.682975e-01,2.989306


In [167]:
def subset(data, key_val_pairs):
    copy = data.copy()
    
    for key_val_pair in key_val_pairs:
        key, val = key_val_pair
        copy = copy[copy[key] == val]
    
    return copy

In [168]:
pfsa_corr = df[df['Grammar type'] == 'pfsa'].corr(numeric_only=True)
pcfg_corr = df[df['Grammar type'] == 'pcfg'].corr(numeric_only=True)

In [169]:
def plot_regs(data, x, y, title_pref):
    
    if y == 'Entropy':
        x, y = y, x
    
    x_axis_min = data[x].min()
    y_axis_min = data[y].min()
    x_axis_max = data[x].max()
    y_axis_max = data[y].max()
    x_padding = (x_axis_max - x_axis_min) * 0.05
    y_padding = (y_axis_max - y_axis_min) * 0.05
    x_axis_range = [x_axis_min - x_padding, x_axis_max + x_padding]
    y_axis_range = [y_axis_min - y_padding, y_axis_max + y_padding]

    if 'PCFG' in title_pref:
        colorbar_title = 'Log # Non-terms'
        x_title = x.replace(
            'Log number of states or non-terminals',
            'Log # non-terms'
        )
        y_title = y.replace(
            'Log number of states or non-terminals',
            'Log # non-terms'
        )
    else:
        colorbar_title = 'Log # States'
        x_title = x.replace(
            'Log number of states or non-terminals',
            'Log # states'
        )
        y_title = y.replace(
            'Log number of states or non-terminals',
            'Log # states'
        )
    
    corr_x_y = pcfg_corr[x][y] if 'PCFG' in title_pref else pfsa_corr[x][y]

    fig = px.scatter(
        data,
        x,
        y,
        symbol='Architecture',
        size='Vocabulary size',
        title=title_pref + f'{y_title} vs. {x_title} (r~{corr_x_y:.2f})',
        color='Log number of states or non-terminals',
        opacity=0.3
    ).update_layout(
        autosize=False,
        width=600,
        height=450,
        margin=dict(l=70, r=120, t=60, b=60),
        xaxis=dict(range=x_axis_range),
        yaxis=dict(range=y_axis_range),
        legend=dict(
            orientation='v',
            yanchor='top',
            y=0.38,         # sits just below where colorbar ends
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='rgba(0, 0, 0, 0.2)',
            borderwidth=1,
            font=dict(size=11),
        ),
        hovermode='closest',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=13)
    ).update_coloraxes(
        colorscale='RdYlBu_r',
        cmin=data['Log number of states or non-terminals'].min(),
        cmax=data['Log number of states or non-terminals'].max(),
        colorbar=dict(
            title=dict(text=colorbar_title, font=dict(size=12)),
            tickfont=dict(size=10),
            x=1.0,
            y=1.04,
            len=0.6,        # takes up top 60% of plot height
            thickness=15,
            yanchor='top',
        )
    )
    
    if 'CE' in y and 'Token-wise' in x:
        endpoint = max(x_axis_range[1], y_axis_range[1])
        fig.add_shape(
            type='line',
            x0=0, y0=0,
            x1=endpoint, y1=endpoint,
            line=dict(color='gray', width=1.5, dash='dash'),
            layer='below'
        )
        
    fig.add_annotation(
        text="Marker size:",
        xref="paper", yref="paper",
        x=1.02, y=0.05,
        showarrow=False,
        font=dict(size=12, color="gray"),
        xanchor='left'
    )
    fig.add_annotation(
        text="Vocab=1K, 5K",
        xref="paper", yref="paper",
        x=1.02, y=0.0,
        showarrow=False,
        font=dict(size=12, color="gray"),
        xanchor='left'
    )
    
    fig.show()

In [170]:
done = set()

for i, x in enumerate(pfsa_corr.columns):
    for y in pfsa_corr.columns[i+1:]:
        if (y, x) in done or (x, y) in done:
            continue
        if 'Seed' in (x, y) or 'Vocabulary size' in (x, y) or 'p-value' in x or 'p-value' in y:
            continue
        if y == 'Token-wise entropy':
            x, y = y, x
        elif y == 'Entropy':
            x, y = y, x
        done.add((x, y))
        done.add((y, x))
        plot_regs(
            subset(
                df,
                [('Grammar type', 'pfsa')]
            ), x, y,
            title_pref = 'PFSAs: '
        )

        plot_regs(
            subset(
                df,
                [('Grammar type', 'pcfg')]
            ), x, y,
            title_pref = 'PCFGs: '
        )